# 📘 Data Dictionary: Feature Engineering Report

Dưới đây là danh sách các thuộc tính (features) đã được khởi tạo và xử lý cho bài toán **Dự báo giá & Nhu cầu theo tuần**. Dữ liệu được tổng hợp ở mức độ: `Product_ID` x `Year_Week`.

### 1. Thông tin Định danh & Thời gian (Context)
*Các biến này xác định ngữ cảnh của dữ liệu.*
* **`Year_Week`**: Mã định danh tuần (Ví dụ: `2023-W01`). Dùng để xác định chu kỳ thời gian.
* **`Week_Start_Date`**: Ngày thứ Hai bắt đầu của tuần đó.
* **`Product_ID`**: Mã sản phẩm duy nhất.
* **`Category` / `Brand_VN`**: Thông tin phân loại sản phẩm (Dùng để nhóm các sản phẩm có hành vi giá tương đồng).

### 2. Biến Mục tiêu & Kết quả (Target & Ground Truth)
*Đây là các biến chúng ta muốn dự báo hoặc tối ưu hóa.*
* **`Qty_Sold`**: (Target Variable) Tổng số lượng sản phẩm bán ra trong tuần hiện tại.
* **`Revenue_Realized`**: Doanh thu ước tính thực tế (`Qty_Sold` * `Selling_Price`).

### 3. Nhóm Biến Giá & Khuyến mãi (Price & Promo Signals)
*Các yếu tố tác động trực tiếp đến quyết định mua hàng (Elasticity Factors).*
* **`Base_Price`**: Giá niêm yết gốc của sản phẩm (Lấy từ `Price_VND`).
* **`Selling_Price`**: Giá bán trung bình thực tế trong tuần (đã bao gồm giảm giá nếu có).
* **`Is_Promo`**: Cờ báo hiệu khuyến mãi (1: Tuần có sự kiện giảm giá, 0: Không).
* **`Discount_Depth`**: Độ sâu giảm giá.
  * *Công thức:* $1 - (\text{Selling\_Price} / \text{Base\_Price})$
  * *Ý nghĩa:* Đo lường mức độ hấp dẫn của giá (VD: 0.2 nghĩa là giảm 20%).
* **`Price_Change_Rate`**: Tỷ lệ biến động giá so với tuần trước.
  * *Công thức:* $(\text{Price}_t - \text{Price}_{t-1}) / \text{Price}_{t-1}$
  * *Ý nghĩa:* Cho biết giá đang tăng hay giảm để bắt hành vi nhạy cảm giá.

### 4. Nhóm Biến Lịch sử Bán hàng (Sales Lag Features)
*Dùng quá khứ để dự đoán tương lai (Time-series Autoregression).*
* **`Qty_Sold_Lag1`**: Số lượng bán được ở tuần liền trước ($t-1$).
* **`Qty_Sold_Avg_4W`**: Sức mua trung bình trong 4 tuần gần nhất (Moving Average).
  * *Ý nghĩa:* Thể hiện xu hướng (Trend) dài hạn hơn, loại bỏ nhiễu của từng tuần lẻ.
* **`Sales_Velocity`**: Gia tốc bán hàng.
  * *Công thức:* $(\text{Lag}_1 - \text{Lag}_2) / \text{Lag}_2$
  * *Ý nghĩa:* Đo tốc độ tăng trưởng. Nếu dương lớn -> Sản phẩm đang "hot", có thể tăng giá.

### 5. Nhóm Biến Hành vi Người dùng (Leading Indicators)
*Dữ liệu tương tác từ tuần trước ($t-1$) dùng để dự báo nhu cầu tuần này. Đây là lợi thế cạnh tranh của mô hình.*
* **`Views_Last_1W`**: Tổng số lượt xem (View AR/Catalog) của tuần trước. -> *Đo lường mức độ quan tâm (Awareness).*
* **`TryOn_Last_1W`**: Tổng số lượt thử đồ ảo (Virtual Try-on) của tuần trước. -> *Đo lường ý định mua (Interest).*
* **`Cart_Adds_Last_1W`**: Tổng số lượt thêm vào giỏ hàng của tuần trước. -> *Đo lường ý định mua sát sườn (High Intent).*

In [26]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 1. Load dữ liệu và chuẩn hóa cột

In [28]:
products_df = pd.read_csv('/content/drive/MyDrive/Output (Sentio)/processed_products_VN_final.csv')
sales_df = pd.read_csv('/content/drive/MyDrive/Output (Sentio)/train_data_forecast.csv')
price_history_df = pd.read_csv('/content/drive/MyDrive/Output (Sentio)/fact_price_history_full.csv')
interactions_df = pd.read_csv('/content/drive/MyDrive/Output (Sentio)/fact_user_interactions.csv')

# Chuẩn hóa tên cột
products_df.columns = products_df.columns.str.strip()
sales_df.rename(columns={'Product ID': 'Product_ID'}, inplace=True)
price_history_df.rename(columns={'Product ID': 'Product_ID'}, inplace=True)
products_df.rename(columns={'Product ID': 'Product_ID'}, inplace=True)

# Convert datetime
sales_df['Sales_Date'] = pd.to_datetime(sales_df['Sales_Date'])
price_history_df['Start_Date'] = pd.to_datetime(price_history_df['Start_Date'])
price_history_df['End_Date'] = pd.to_datetime(price_history_df['End_Date'])
interactions_df['Timestamp'] = pd.to_datetime(interactions_df['Timestamp'])

# 2. Tạo cột thời gian

In [29]:
def get_week_info(df, date_col):
    # Lấy ngày đầu tuần (Thứ 2)
    df['Week_Start_Date'] = df[date_col] - pd.to_timedelta(df[date_col].dt.weekday, unit='D')
    df['Week_Start_Date'] = df['Week_Start_Date'].dt.normalize()
    # Tạo cột Year_Week (VD: 2023-W01)
    df['Year_Week'] = df['Week_Start_Date'].dt.strftime('%Y-W%U')
    return df

In [30]:
sales_df = get_week_info(sales_df, 'Sales_Date')
interactions_df = get_week_info(interactions_df, 'Timestamp')

# 3. Xử lý price_history (gom theo ngày -> theo tuần)

In [31]:
daily_prices = []
for _, row in price_history_df.iterrows():
    date_range = pd.date_range(start=row['Start_Date'], end=row['End_Date'], freq='D')
    temp_df = pd.DataFrame({
        'Date': date_range,
        'Product_ID': row['Product_ID'],
        'Selling_Price': row['Price'],
        'Is_Promo': row['Is_Promo']
    })
    daily_prices.append(temp_df)

In [32]:
df_daily_price = pd.concat(daily_prices, ignore_index=True)
df_daily_price = get_week_info(df_daily_price, 'Date')

In [33]:
# Gom nhóm giá theo tuần
df_weekly_price = df_daily_price.groupby(['Product_ID', 'Year_Week', 'Week_Start_Date']).agg({
    'Selling_Price': 'mean',
    'Is_Promo': 'max'
}).reset_index()

# 4. Xử lý tương tác (theo từng Action)

In [34]:
# Pivot table để đếm riêng từng loại hành động
df_interactions_pivot = interactions_df.pivot_table(
    index=['Product_ID', 'Year_Week', 'Week_Start_Date'],
    columns='Action_Type',
    values='Interaction_ID',
    aggfunc='count',
    fill_value=0
).reset_index()

In [35]:
# Đổi tên cột cho rõ ràng
df_interactions_pivot.rename(columns={
    'VIEW_AR_CATALOG': 'Count_View',
    'SCAN_STOREFRONT': 'Count_Scan',
    'TRY_ON_VIRTUAL': 'Count_TryOn',
    'ADD_TO_CART': 'Count_AddToCart'
}, inplace=True)

In [36]:
# Tính tổng Weighted Score
ACTION_WEIGHTS = {'Count_Scan': 1, 'Count_View': 2, 'Count_TryOn': 5, 'Count_AddToCart': 8}
df_interactions_pivot['Interaction_Score'] = (
    df_interactions_pivot['Count_Scan'] * 1 +
    df_interactions_pivot['Count_View'] * 2 +
    df_interactions_pivot['Count_TryOn'] * 5 +
    df_interactions_pivot['Count_AddToCart'] * 8
)

# 5. Xử lý bán hàng (sales)

In [37]:
df_weekly_sales = sales_df.groupby(['Product_ID', 'Year_Week', 'Week_Start_Date']).agg({
    'Quantity': 'sum'
}).reset_index().rename(columns={'Quantity': 'Qty_Sold'})

# 6. Merge dữ liệu và tạo feature

In [38]:
master_df = df_weekly_price.copy()
master_df = master_df.merge(df_weekly_sales, on=['Product_ID', 'Year_Week', 'Week_Start_Date'], how='left')
master_df = master_df.merge(df_interactions_pivot, on=['Product_ID', 'Year_Week', 'Week_Start_Date'], how='left')
master_df = master_df.merge(products_df[['Product_ID', 'Category', 'Brand_VN', 'Price_VND']], on='Product_ID', how='left')
master_df.rename(columns={'Price_VND': 'Base_Price'}, inplace=True)

In [39]:
# Fill NaN
cols_to_fill = ['Qty_Sold', 'Count_View', 'Count_Scan', 'Count_TryOn', 'Count_AddToCart', 'Interaction_Score']
master_df[cols_to_fill] = master_df[cols_to_fill].fillna(0)

## Feature Engineering

In [40]:
master_df = master_df.sort_values(['Product_ID', 'Week_Start_Date'])

In [41]:
def create_scenario_features(df):

    # 1. NHÓM SALES HISTORY & REVENUE
    # Qty_Sold_Last_1W (Lag 1)
    df['Qty_Sold_Lag1'] = df.groupby('Product_ID')['Qty_Sold'].shift(1)
    # Qty_Sold_Lag2 (Để tính Velocity)
    df['Qty_Sold_Lag2'] = df.groupby('Product_ID')['Qty_Sold'].shift(2)
    # Sales_Velocity: (Tốc độ tăng trưởng so với 2 tuần trước)
    # Công thức: (Lag1 - Lag2) / (Lag2 + 1)
    df['Sales_Velocity'] = (df['Qty_Sold_Lag1'] - df['Qty_Sold_Lag2']) / (df['Qty_Sold_Lag2'] + 1)

    # Qty_Sold_Avg_4W (Trend)
    df['Qty_Sold_Avg_4W'] = df.groupby('Product_ID')['Qty_Sold'].shift(1).rolling(4).mean()

    # Revenue_Last_1W (Doanh thu tuần trước)
    df['Revenue_Realized'] = df['Qty_Sold'] * df['Selling_Price']
    df['Revenue_Last_1W'] = df.groupby('Product_ID')['Revenue_Realized'].shift(1)

    # 2. NHÓM PRICE & PROMO
    # Avg_Price_Last_1W
    df['Avg_Price_Last_1W'] = df.groupby('Product_ID')['Selling_Price'].shift(1)
    # Discount_Depth (Độ sâu giảm giá hiện tại)
    df['Discount_Depth'] = 1 - (df['Selling_Price'] / df['Base_Price'])
    # Price_Change_Rate (Biến động giá tuần này so với tuần trước)
    df['Price_Change_Rate'] = (df['Selling_Price'] - df['Avg_Price_Last_1W']) / df['Avg_Price_Last_1W']
    # Is_Promo_Next_Week (Biến mục tiêu cho tương lai hoặc ngữ cảnh hiện tại)
    # Ở đây ta giữ Is_Promo của dòng hiện tại làm context

    # 3. NHÓM HÀNH VI (LEADING INDICATORS)
    # Các biến này cần Lag 1 (Hành vi tuần trước dự báo Sale tuần này)
    df['Views_Last_1W'] = df.groupby('Product_ID')['Count_View'].shift(1)
    df['TryOn_Last_1W'] = df.groupby('Product_ID')['Count_TryOn'].shift(1)
    df['Cart_Adds_Last_1W'] = df.groupby('Product_ID')['Count_AddToCart'].shift(1)
    df['Interaction_Score_Last_1W'] = df.groupby('Product_ID')['Interaction_Score'].shift(1)

    # Interest_Conversion (Tỷ lệ chuyển đổi tuần trước)
    df['Interest_Conversion'] = df['Qty_Sold_Lag1'] / (df['Views_Last_1W'] + 1)

    return df

In [42]:
master_df_final = create_scenario_features(master_df)

In [43]:
# Làm sạch dữ liệu (Bỏ các dòng đầu thiếu Lag)
master_df_final = master_df_final.dropna(subset=['Qty_Sold_Lag2', 'Qty_Sold_Avg_4W'])

In [44]:
# Chọn cột theo đúng thứ tự logic để dễ nhìn
final_columns = [
    'Year_Week', 'Week_Start_Date', 'Product_ID',
    'Category', 'Base_Price', 'Selling_Price', 'Is_Promo', # Info & Context
    'Qty_Sold', 'Revenue_Realized',                        # Target (Tuần này)
    'Qty_Sold_Lag1', 'Qty_Sold_Avg_4W', 'Sales_Velocity',  # Lag Features (Sales)
    'Views_Last_1W', 'TryOn_Last_1W', 'Cart_Adds_Last_1W', # Lag Features (Behavior)
    'Discount_Depth', 'Price_Change_Rate'                  # Price Features
]

In [45]:
output_df = master_df_final[final_columns]

print("-" * 30)
print("DỮ LIỆU ĐÃ HOÀN TẤT VỚI ĐẦY ĐỦ CÁC CỘT YÊU CẦU:")
print(output_df.info())

------------------------------
DỮ LIỆU ĐÃ HOÀN TẤT VỚI ĐẦY ĐỦ CÁC CỘT YÊU CẦU:
<class 'pandas.core.frame.DataFrame'>
Index: 10200 entries, 4 to 10399
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Year_Week          10200 non-null  object        
 1   Week_Start_Date    10200 non-null  datetime64[ns]
 2   Product_ID         10200 non-null  int64         
 3   Category           10200 non-null  object        
 4   Base_Price         10200 non-null  int64         
 5   Selling_Price      10200 non-null  float64       
 6   Is_Promo           10200 non-null  int64         
 7   Qty_Sold           10200 non-null  float64       
 8   Revenue_Realized   10200 non-null  float64       
 9   Qty_Sold_Lag1      10200 non-null  float64       
 10  Qty_Sold_Avg_4W    10200 non-null  float64       
 11  Sales_Velocity     10200 non-null  float64       
 12  Views_Last_1W      10200 non-null  float64

In [46]:
output_df

,Year_Week,Week_Start_Date,Product_ID,Category,Base_Price,Selling_Price,Is_Promo,Qty_Sold,Revenue_Realized,Qty_Sold_Lag1,Qty_Sold_Avg_4W,Sales_Velocity,Views_Last_1W,TryOn_Last_1W,Cart_Adds_Last_1W,Discount_Depth,Price_Change_Rate
4,2022-W04,2022-01-24,4,Feminine,799000,674000.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.156446,0.0
5,2022-W05,2022-01-31,4,Feminine,799000,674000.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.156446,0.0
6,2022-W06,2022-02-07,4,Feminine,799000,674000.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.156446,0.0
7,2022-W07,2022-02-14,4,Feminine,799000,674000.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.156446,0.0
8,2022-W08,2022-02-21,4,Feminine,799000,674000.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.156446,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10395,2025-W46,2025-11-17,249,Masculine,399000,418950.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.050000,0.0
10396,2025-W47,2025-11-24,249,Masculine,399000,418950.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.050000,0.0
10397,2025-W48,2025-12-01,249,Masculine,399000,418950.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.050000,0.0
10398,2025-W49,2025-12-08,249,Masculine,399000,418950.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.050000,0.0


In [47]:
output_df.to_csv('/content/drive/MyDrive/Output (Sentio)/weekly_data_final.csv', index=False)